# Problem Statement : Hospital Patient Data Analysis
### Context:
### A hospital maintains patient records including admission details, department, diagnosis, doctor, and bill amount. You have two datasets: one with patient info and another with billing details. Some patients have blank bill amounts, and there are multiple rows for the same patient due to follow-ups.
## Tasks:
### 1.	Load the patient dataset and show summary with info().
### 2.	Select only the columns relevant for billing: ['PatientID', 'Department', 'Doctor', 'BillAmount'].
### 3.	Drop administrative columns like ['ReceptionistID', 'CheckInTime'].
### 4.	Use groupby to find total bill amount per department.
### 5.	Remove duplicate patient records based on PatientID.
### 6.	Fill missing BillAmount values with the mean bill amount.
### 7.	Merge the billing dataset with patient dataset on PatientID.
### 8.	Concatenate an additional DataFrame that contains new patients for the current week (row-wise).
### 9.	Concatenate new billing category columns like ['InsuranceCovered', 'FinalAmount'] (column-wise).


In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv('Patient_Data.csv')

In [3]:
# 1. Load the patient dataset and show summary with info().
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6 entries, 0 to 5
Data columns (total 7 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   PatientID       6 non-null      int64  
 1   Name            6 non-null      object 
 2   Department      6 non-null      object 
 3   Doctor          6 non-null      object 
 4   BillAmount      4 non-null      float64
 5   ReceptionistID  6 non-null      int64  
 6   CheckInTime     6 non-null      object 
dtypes: float64(1), int64(2), object(4)
memory usage: 468.0+ bytes


In [4]:
# 2. Select only the columns relevant for billing:['PatientID', 'Department', 'Doctor', 'BillAmount'].
patients = df[['PatientID','Department','Doctor','BillAmount']]
patients

,PatientID,Department,Doctor,BillAmount
0,101,Cardiology,Dr. Smith,5000.0
1,102,Neurology,Dr. John,NaN
2,103,Orthopedics,Dr. Lee,7500.0
3,104,Cardiology,Dr. Smith,6200.0
4,105,Dermatology,Dr. Rose,NaN
5,101,Cardiology,Dr. Smith,5000.0


In [5]:
# 3. Drop administrative columns like ['ReceptionistID', 'CheckInTime'].
patients = df.drop(columns=['ReceptionistID','CheckInTime'])
patients

,PatientID,Name,Department,Doctor,BillAmount
0,101,Alice,Cardiology,Dr. Smith,5000.0
1,102,Bob,Neurology,Dr. John,NaN
2,103,Charlie,Orthopedics,Dr. Lee,7500.0
3,104,David,Cardiology,Dr. Smith,6200.0
4,105,Eva,Dermatology,Dr. Rose,NaN
5,101,Alice,Cardiology,Dr. Smith,5000.0


In [6]:
# 4.	Use groupby to find total bill amount per department.
dept_bill = patients.groupby('Department')['BillAmount'].sum().reset_index(name='Total Bill')
print(dept_bill)

    Department  Total Bill
0   Cardiology     16200.0
1  Dermatology         0.0
2    Neurology         0.0
3  Orthopedics      7500.0


In [7]:
# 5. Remove duplicate patients
patients = df.drop_duplicates(subset='PatientID')
patients

,PatientID,Name,Department,Doctor,BillAmount,ReceptionistID,CheckInTime
0,101,Alice,Cardiology,Dr. Smith,5000.0,1,2023-01-10 09:00
1,102,Bob,Neurology,Dr. John,NaN,2,2023-01-11 10:30
2,103,Charlie,Orthopedics,Dr. Lee,7500.0,1,2023-01-12 11:00
3,104,David,Cardiology,Dr. Smith,6200.0,3,2023-01-13 12:00
4,105,Eva,Dermatology,Dr. Rose,NaN,2,2023-01-14 08:45


In [8]:
# 6. Fill missing BillAmount values
mean = df['BillAmount'].mean()
df['BillAmount'].fillna(mean)

0    5000.0
1    5925.0
2    7500.0
3    6200.0
4    5925.0
5    5000.0
Name: BillAmount, dtype: float64

In [9]:
# 7. Merge billing dataset
billing = pd.read_csv("Billing_Data.csv")
merged_data = pd.merge(patients, billing, on='PatientID')
merged_data

,PatientID,Name,Department,Doctor,BillAmount,ReceptionistID,CheckInTime,InsuranceCovered,FinalAmount
0,101,Alice,Cardiology,Dr. Smith,5000.0,1,2023-01-10 09:00,2000,3000
1,102,Bob,Neurology,Dr. John,NaN,2,2023-01-11 10:30,1500,3500
2,103,Charlie,Orthopedics,Dr. Lee,7500.0,1,2023-01-12 11:00,2500,5000
3,104,David,Cardiology,Dr. Smith,6200.0,3,2023-01-13 12:00,3000,3200
4,105,Eva,Dermatology,Dr. Rose,NaN,2,2023-01-14 08:45,1000,4000


In [15]:
#  8. Concatenate new patients (row-wise)


# Example new patients data for current week
new_patients = pd.DataFrame({
    'PatientID': [106, 107],
    'Name': ['mama','mami'],
    'Department': ['Cardiology', 'Neurology'],
    'Doctor': ['Dr. Rao', 'Dr. Mehta'],
    'BillAmount': [5000, 6500],
    'ReceptionistID': [2, 1]
})

# Concatenate with existing merged dataset
final_data = pd.concat([merged_data, new_patients], axis=0)

print(final_data)

   PatientID     Name   Department     Doctor  BillAmount  ReceptionistID  \
0        101    Alice   Cardiology  Dr. Smith      5000.0               1   
1        102      Bob    Neurology   Dr. John         NaN               2   
2        103  Charlie  Orthopedics    Dr. Lee      7500.0               1   
3        104    David   Cardiology  Dr. Smith      6200.0               3   
4        105      Eva  Dermatology   Dr. Rose         NaN               2   
0        106     mama   Cardiology    Dr. Rao      5000.0               2   
1        107     mami    Neurology  Dr. Mehta      6500.0               1   

        CheckInTime  InsuranceCovered  FinalAmount  
0  2023-01-10 09:00            2000.0       3000.0  
1  2023-01-11 10:30            1500.0       3500.0  
2  2023-01-12 11:00            2500.0       5000.0  
3  2023-01-13 12:00            3000.0       3200.0  
4  2023-01-14 08:45            1000.0       4000.0  
0               NaN               NaN          NaN  
1           

In [20]:
# 9.Add new billing columns (column-wise)
# Create new billing DataFrame
new_billing_columns = pd.DataFrame({
    'InsuranceCovered': final_data['InsuranceCovered'],
    'FinalAmount': final_data['BillAmount'] - final_data['InsuranceCovered']
})

# Concatenate column-wise
final_data = pd.concat([final_data, new_billing_columns], axis=1)

# View result
print(final_data.head())

   PatientID     Name   Department     Doctor  BillAmount  ReceptionistID  \
0        101    Alice   Cardiology  Dr. Smith      5000.0               1   
1        102      Bob    Neurology   Dr. John         0.0               2   
2        103  Charlie  Orthopedics    Dr. Lee      7500.0               1   
3        104    David   Cardiology  Dr. Smith      6200.0               3   
4        105      Eva  Dermatology   Dr. Rose         0.0               2   

        CheckInTime  InsuranceCovered  FinalAmount  Total_Bill     Tax  \
0  2023-01-10 09:00            2000.0       3000.0      5000.0   900.0   
1  2023-01-11 10:30            1500.0       3500.0         0.0     0.0   
2  2023-01-12 11:00            2500.0       5000.0      7500.0  1350.0   
3  2023-01-13 12:00            3000.0       3200.0      6200.0  1116.0   
4  2023-01-14 08:45            1000.0       4000.0         0.0     0.0   

   Amount_After_Insurance  InsuranceCovered  FinalAmount  
0                  3000.0        

In [12]:
# New billing columns
billing_columns = pd.DataFrame({
    'InsuranceCovered': ['Yes', 'No', 'Yes', 'No'],
    'FinalAmount': [4500, 6000, 5000, 6500]
})

# Concatenate column-wise
final_data = pd.concat([final_data, billing_columns], axis=1)

print(final_data)

   PatientID     Name   Department     Doctor  BillAmount  ReceptionistID  \
0        101    Alice   Cardiology  Dr. Smith      5000.0               1   
1        102      Bob    Neurology   Dr. John         0.0               2   
2        103  Charlie  Orthopedics    Dr. Lee      7500.0               1   
3        104    David   Cardiology  Dr. Smith      6200.0               3   
4        105      Eva  Dermatology   Dr. Rose         0.0               2   
0        106     mama   Cardiology    Dr. Rao      5000.0               2   
1        107     mami    Neurology  Dr. Mehta      6500.0               1   

        CheckInTime InsuranceCovered  FinalAmount  Total_Bill     Tax  \
0  2023-01-10 09:00           2000.0       3000.0      5000.0   900.0   
1  2023-01-11 10:30           1500.0       3500.0         0.0     0.0   
2  2023-01-12 11:00           2500.0       5000.0      7500.0  1350.0   
3  2023-01-13 12:00           3000.0       3200.0      6200.0  1116.0   
4  2023-01-14 08:4